# Combine Seed Plots

This notebook iterates through a specific folder structure to combine 4 corresponding PNG images vertically. Images are expected to be in folders that end with a number from `0` to `3`.

In [1]:
import os
import glob
from PIL import Image
from collections import defaultdict

# --- Paths ---
INPUT_BASE_DIR = r"C:\Users\franc\local_files(non_ODrive)\protPred_colab_runs\AlaRep\plotting_input_and_output\plots\TM_Score"
OUTPUT_DIR = r"C:\Users\franc\local_files(non_ODrive)\protPred_colab_runs\AlaRep\plotting_input_and_output\plots\combined_plots_for_seeds"


In [2]:
# Create output directory if it doesn't exist
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Group folders by their base name (excluding the last character)
folder_groups = defaultdict(dict)

for folder_name in os.listdir(INPUT_BASE_DIR):
    folder_path = os.path.join(INPUT_BASE_DIR, folder_name)
    if os.path.isdir(folder_path):
        # We expect the last character to be a number 0-3
        base_name = folder_name[:-1]
        try:
            seed_idx = int(folder_name[-1])
            folder_groups[base_name][seed_idx] = folder_path
        except ValueError:
            pass # ignore folders that don't end in a number


In [3]:
# Process each group
for base_name, paths_dict in folder_groups.items():
    # Only process if we have exactly 4 folders (0, 1, 2, 3)
    if set(paths_dict.keys()) == {0, 1, 2, 3}:
        images = []
        valid = True
        
        # Load images in order 0 to 3
        for i in range(4):
            folder_path = paths_dict[i]
            # Find the single .png file in the folder
            png_files = glob.glob(os.path.join(folder_path, "*.png"))
            if len(png_files) != 1:
                print(f"Warning: Expected exactly 1 PNG file in {folder_path}, found {len(png_files)}. Skipping group.")
                valid = False
                break
            
            img_path = png_files[0]
            try:
                img = Image.open(img_path)
                images.append(img)
            except Exception as e:
                print(f"Error opening image {img_path}: {e}")
                valid = False
                break
                
        if not valid:
            continue
            
        # Combine images vertically
        widths, heights = zip(*(i.size for i in images))
        
        max_width = max(widths)
        total_height = sum(heights)
        
        new_im = Image.new('RGB', (max_width, total_height), color='white')
        
        y_offset = 0
        for im in images:
            new_im.paste(im, (0, y_offset))
            y_offset += im.size[1]
            
        # Determine output filename
        # Naming of the output files: the 4-times repeating folder names up to (excluding) the 4th occurence of an underscore "_"
        # e.g., qu_mask_15_ASCT2_AlaRepSEEDchange -> qu_mask_15_ASCT2
        parts = base_name.split("_")
        if len(parts) > 4:
            out_name = "_".join(parts[:4])
        else:
            out_name = base_name
            
        out_path = os.path.join(OUTPUT_DIR, f"{out_name}.png")
        new_im.save(out_path)
        print(f"Saved combined image to {out_path}")
